# Data set loading

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# ── Load the pre-calculated gaps file ───────────────────────────
gaps = pd.read_csv('all_attainment_gaps.csv')


# ── Setup ────────────────────────────────────────────────────────
os.makedirs('visualisations', exist_ok=True)

UON_UKPRN = 10007154
YEARS = ['2018-19','2019-20','2020-21','2021-22','2022-23','2023-24']

# ── Colour palette ───────────────────────────────────────────────
NAVY   = '#00324A'
TEAL   = '#006A72'
GOLD   = '#C4A747'
RED    = '#C0392B'
GREEN  = '#1A7A4A'
AMBER  = '#D4820A'
SILVER = '#B0BEC5'
LIGHT  = '#F7F8FA'

# Column rename , UoN selection


In [4]:
gaps = pd.read_csv('all_attainment_gaps.csv')
gaps['UKPRN'] = gaps['UKPRN'].astype(str)

# Then update UoN identifier to match
UON_UKPRN = '10007154'  # string not integer

# Print actual columns first
print("Actual columns:", gaps.columns.tolist())

# Rename to standard names the script expects
# Adjust the LEFT side to match what print() showed you
gaps = gaps.rename(columns={
    'Academic Year'  : 'Year',
    'Provider Name'  : 'Provider',
    'Char Code'      : 'Char Code',
    'Attr Code'      : 'Attr Code',
    'Attainment Gap (pp)' : 'Gap',
})

print("Renamed columns:", gaps.columns.tolist())
print(gaps.head(3))

Actual columns: ['UKPRN', 'Provider Name', 'Academic Year', 'Char Code', 'Attr Code', 'Comp Code', 'Target Rate (%)', 'Comparison Rate (%)', 'Attainment Gap (pp)', 'Characteristic', 'Attribute']
Renamed columns: ['UKPRN', 'Provider', 'Year', 'Char Code', 'Attr Code', 'Comp Code', 'Target Rate (%)', 'Comparison Rate (%)', 'Gap', 'Characteristic', 'Attribute']
      UKPRN                                           Provider     Year  \
0  10003270  Imperial College of Science, Technology and Me...  2018-19   
1  10003270  Imperial College of Science, Technology and Me...  2018-19   
2  10003270  Imperial College of Science, Technology and Me...  2018-19   

   Char Code  Attr Code  Comp Code  Target Rate (%)  Comparison Rate (%)  Gap  \
0          1        1.2        1.2             88.1                 88.1  0.0   
1          2        2.4        2.4             90.2                 90.2  0.0   
2          3        3.2        3.5             90.9                 88.2 -2.7   

             

In [5]:
print(gaps.columns.tolist())
print(gaps.head(3))

['UKPRN', 'Provider', 'Year', 'Char Code', 'Attr Code', 'Comp Code', 'Target Rate (%)', 'Comparison Rate (%)', 'Gap', 'Characteristic', 'Attribute']
      UKPRN                                           Provider     Year  \
0  10003270  Imperial College of Science, Technology and Me...  2018-19   
1  10003270  Imperial College of Science, Technology and Me...  2018-19   
2  10003270  Imperial College of Science, Technology and Me...  2018-19   

   Char Code  Attr Code  Comp Code  Target Rate (%)  Comparison Rate (%)  Gap  \
0          1        1.2        1.2             88.1                 88.1  0.0   
1          2        2.4        2.4             90.2                 90.2  0.0   
2          3        3.2        3.5             90.9                 88.2 -2.7   

              Characteristic                 Attribute  
0        Age On Commencement            Young Under 21  
1            Disability Type  No Known Disability Type  
2  English IMD Quintile 2019                     IMDQ2

# Helper functions for easy Visualisations

In [14]:
def get_uon_trend(char_code, attr_code):
    """Pull UoN gap values across all years from the CSV."""
    subset = gaps[
        (gaps['UKPRN'] == UON_UKPRN) &
        (gaps['Char Code'] == char_code) &
        (gaps['Attr Code'] == attr_code)
    ].set_index('Year')
    return [
        float(subset.loc[y, 'Gap']) if y in subset.index else None 
        for y in YEARS
    ]


def get_sector_trend(char_code, attr_code):
    """Pull sector average gap values across all years from the CSV."""
    subset = gaps[
        (gaps['UKPRN'] == 'SECTOR') &
        (gaps['Char Code'] == char_code) &
        (gaps['Attr Code'] == attr_code)
    ].set_index('Year')
    return [
        float(subset.loc[y, 'Gap']) if y in subset.index else None 
        for y in YEARS
    ]


def get_peer_comparison(char_code, attr_code, year='2023-24'):
    """Pull all providers ranked by gap for latest year from the CSV."""
    subset = gaps[
        (gaps['Char Code'] == char_code) &
        (gaps['Attr Code'] == attr_code) &
        (gaps['Year'] == year) &
        (gaps['UKPRN'] != 'SECTOR')
    ].sort_values('Gap', ascending=True).copy()

    # Shorten names for chart readability
    subset['Short Name'] = subset['Provider']\
        .str.replace('University of Newcastle upon Tyne', 'Newcastle')\
        .str.replace('Imperial College of Science, Technology and Medicine', 'Imperial')\
        .str.replace('The London School of Economics and Political Science', 'LSE')\
        .str.replace('Queen Mary University of London', 'Queen Mary')\
        .str.replace('University College London', 'UCL')\
        .str.replace('University of Nottingham, the', 'UoN ★')\
        .str.replace('The University of ', 'Uni of ')\
        .str.replace('University of ', 'Uni of ')

    return subset


def get_sector_gap(char_code, attr_code, year='2023-24'):
    """Pull single sector gap value for a given year."""
    row = gaps[
        (gaps['UKPRN'] == 'SECTOR') &
        (gaps['Char Code'] == char_code) &
        (gaps['Attr Code'] == attr_code) &
        (gaps['Year'] == year)
    ]
    return float(row['Gap'].values[0]) if len(row) > 0 else None


def style_chart(ax, title, xlabel, ylabel=None):
    """Consistent styling across all charts."""
    ax.set_title(title, fontsize=14, fontweight='bold', color=NAVY, pad=15)
    ax.set_xlabel(xlabel, fontsize=11, color=NAVY)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=11, color=NAVY)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#E0E8EE')
    ax.spines['bottom'].set_color('#E0E8EE')
    ax.tick_params(colors='#64788B')
    ax.set_facecolor(LIGHT)
    ax.grid(axis='y', color='#E0E8EE', linewidth=0.8, zorder=0)

# Trend Visualisation 

In [22]:
def plot_trend(char_code, attr_code, title, 
               target_val, line_color, filename):

    uon_vals    = get_uon_trend(char_code, attr_code)
    sector_vals = get_sector_trend(char_code, attr_code)

    fig, ax = plt.subplots(figsize=(10, 5.5))
    fig.patch.set_facecolor(LIGHT)

    x = range(len(YEARS))

    ax.plot(x, sector_vals, color=SILVER, linewidth=2.5,
            linestyle='--', marker='o', markersize=5,
            label='Sector Average', zorder=3)

    ax.plot(x, uon_vals, color=line_color, linewidth=3,
            marker='o', markersize=7, label='UoN', zorder=4)

    # Value labels on UoN points
    for i, val in enumerate(uon_vals):
        if val is not None:
            ax.annotate(f'{val}pp', xy=(i, val),
                        xytext=(0, 12), textcoords='offset points',
                        ha='center', fontsize=9,
                        color=line_color, fontweight='bold')

    # APP target line
    ax.axhline(y=target_val, color=GOLD, linewidth=2,
               linestyle=':', zorder=2,
               label=f'APP Target: {target_val}pp by 2028/29')

    ax.set_xticks(x)
    ax.set_xticklabels(YEARS, fontsize=10)
    ax.set_ylim(0, max(filter(None, uon_vals + sector_vals)) * 1.3)

    style_chart(ax, title,
                xlabel='Academic Year',
                ylabel='Attainment Gap (percentage points)')

    ax.legend(loc='upper left', framealpha=0.9, fontsize=10)
    plt.tight_layout()
    plt.savefig(f'visualisations/{filename}', dpi=150,
                bbox_inches='tight', facecolor=LIGHT)
    plt.close()
    print(f'Saved: {filename}')


plot_trend(4, 4.2,
    title='Black–White Attainment Gap: UoN vs Sector',
    target_val=14.3, line_color=RED,
    filename='01_trend_black_white.png')

plot_trend(4, 4.1,
    title='Asian–White Attainment Gap: UoN vs Sector',
    target_val=7.0, line_color=AMBER,
    filename='02_trend_asian_white.png')

plot_trend(1, 1.1,
    title='Mature–Young Attainment Gap: UoN vs Sector',
    target_val=7.0, line_color=GREEN,
    filename='03_trend_mature_young.png')

plot_trend(5, 5.1,
    title='FSM Eligible Attainment Gap: UoN vs Sector',
    target_val=9.0, line_color=NAVY,
    filename='04_trend_fsm.png')

Saved: 01_trend_black_white.png
Saved: 02_trend_asian_white.png
Saved: 03_trend_mature_young.png
Saved: 04_trend_fsm.png


# Comparison Visualisation


In [15]:
def plot_peer_comparison(char_code, attr_code, title, 
                         bar_color, filename):

    df = get_peer_comparison(char_code, attr_code)
    sector_val = get_sector_gap(char_code, attr_code)

    colors = [RED if 'UoN ★' in str(name) else bar_color
              for name in df['Short Name']]

    fig, ax = plt.subplots(figsize=(10, 7))
    fig.patch.set_facecolor(LIGHT)

    ax.barh(df['Short Name'], df['Gap'],
            color=colors, edgecolor='white',
            linewidth=0.5, height=0.7, zorder=3)

    # Value labels
    for _, row in df.iterrows():
        ax.text(row['Gap'] + 0.3,
                list(df['Short Name']).index(row['Short Name']),
                f"{row['Gap']}pp",
                va='center', fontsize=9,
                color=NAVY, fontweight='bold')

    # Sector average line
    if sector_val:
        ax.axvline(x=sector_val, color=GOLD, linewidth=2,
                   linestyle='--',
                   label=f'Sector Average: {sector_val}pp')
        ax.legend(fontsize=10, framealpha=0.9)

    style_chart(ax, title,
                xlabel='Attainment Gap (percentage points)')
    ax.grid(axis='x', color='#E0E8EE', linewidth=0.8, zorder=0)
    ax.grid(axis='y', visible=False)

    plt.tight_layout()
    plt.savefig(f'visualisations/{filename}', dpi=150,
                bbox_inches='tight', facecolor=LIGHT)
    plt.close()
    print(f'Saved: {filename}')


plot_peer_comparison(4, 4.2,
    title='Black–White Gap: All Providers Ranked (2023-24)',
    bar_color=TEAL, filename='05_peers_black_white.png')

plot_peer_comparison(4, 4.1,
    title='Asian–White Gap: All Providers Ranked (2023-24)',
    bar_color=TEAL, filename='06_peers_asian_white.png')

plot_peer_comparison(1, 1.1,
    title='Mature–Young Gap: All Providers Ranked (2023-24)',
    bar_color=TEAL, filename='07_peers_mature_young.png')

plot_peer_comparison(5, 5.1,
    title='FSM Eligible Gap: All Providers Ranked (2023-24)',
    bar_color=TEAL, filename='08_peers_fsm.png')

Saved: 05_peers_black_white.png
Saved: 06_peers_asian_white.png
Saved: 07_peers_mature_young.png
Saved: 08_peers_fsm.png


# Progress Comparisons 

In [24]:
def plot_target_progress(label, baseline, latest, 
                         target, color, filename):

    fig, ax = plt.subplots(figsize=(9, 4))
    fig.patch.set_facecolor(LIGHT)

    categories = ['OfS Baseline\n(2021-22)',
                  'Latest Data\n(2023-24)',
                  'APP Target\n(2028-29)']
    values     = [baseline, latest, target]
    colors     = [SILVER, color, GREEN]

    bars = ax.bar(categories, values, color=colors,
                  width=0.4, zorder=3, edgecolor='white')

    # Value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                f'{val}pp', ha='center',
                fontsize=13, fontweight='bold', color=NAVY)

    ax.set_ylim(0, max(values) * 1.35)

    style_chart(ax,
                title=f'{label}: Baseline vs Current vs Target',
                xlabel='',
                ylabel='Gap (percentage points)')
    ax.grid(axis='x', visible=False)

    plt.tight_layout()
    plt.savefig(f'visualisations/{filename}', dpi=150,
                bbox_inches='tight', facecolor=LIGHT)
    plt.close()
    print(f'Saved: {filename}')


plot_target_progress('Black–White Awarding Gap',
    baseline=18.2, latest=28.6, target=14.3,
    color=RED,   filename='09_target_black_white.png')

plot_target_progress('Asian–White Awarding Gap',
    baseline=7.1,  latest=11.7, target=7.0,
    color=AMBER, filename='10_target_asian_white.png')

plot_target_progress('Mature–Young Awarding Gap',
    baseline=15.8, latest=1.0,  target=7.0,
    color=GREEN, filename='11_target_mature_young.png')

plot_target_progress('FSM Eligible Awarding Gap',
    baseline=12.0, latest=14.0, target=9.0,
    color=RED,   filename='12_target_fsm.png')

print('\nAll done — charts saved to /visualisations folder')

Saved: 09_target_black_white.png
Saved: 10_target_asian_white.png
Saved: 11_target_mature_young.png
Saved: 12_target_fsm.png

All done — charts saved to /visualisations folder
